# Week 4 — live coding (completed)

**Social and Cultural Analytics (7AAVBCS5)**


## Setup


In [ ]:
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

DATA_REPO = "https://github.com/markjhill/7AAVBCS5-data.git"


def data_path(name: str) -> Path:
    """Return a usable path to a file or folder in the module's data folder."""
    here = Path.cwd()
    for candidate in (here, *here.parents):
        local = candidate / "data" / name
        if local.exists():
            return local

    downloaded = Path("7AAVBCS5-data")
    if not downloaded.is_dir():
        print("No local data folder found -- downloading the module data (once)...")
        subprocess.run(
            ["git", "clone", "--depth", "1", DATA_REPO, str(downloaded)],
            check=True,
        )
    return downloaded / name


## Exercise 1: quantify missingness


In [ ]:
venues = pd.read_csv(data_path("london_cultural_venues_2023.csv"))
venues.head()


In [ ]:
print(venues.shape)
print(list(venues.columns))
venues.info()


In [ ]:
missing_summary = pd.DataFrame({
    "missing": venues.isna().sum(),
    "missing_%": (venues.isna().mean() * 100).round(1),
})
missing_summary.sort_values("missing", ascending=False)


In [ ]:
missing_summary.sort_values("missing", ascending=False).head(3)


## Exercise 2: missingness indicators


In [ ]:
venues = venues.copy()
venues["has_website"] = venues["website"].notna()
venues["missing_postcode"] = venues["postcode"].isna()
venues[["name", "has_website", "missing_postcode"]].head()


In [ ]:
print(venues["has_website"].value_counts())
print(venues["missing_postcode"].value_counts())


In [ ]:
website_by_type = (
    venues.groupby("venue_type")
    .agg(venues=("venue_id", "size"), website_share=("has_website", "mean"))
)
website_by_type["website_share"] = (website_by_type["website_share"] * 100).round(1)
website_by_type.sort_values("website_share").head(10)


In [ ]:
venues["venue_type"].value_counts().head(10)


## Exercise 3: borough patterns


In [ ]:
website_by_borough = (
    venues.groupby("borough_name")
    .agg(
        venues=("venue_id", "size"),
        website_share=("has_website", "mean"),
        postcode_missing=("missing_postcode", "mean"),
    )
)
website_by_borough[["website_share", "postcode_missing"]] = (
    website_by_borough[["website_share", "postcode_missing"]] * 100
).round(1)
website_by_borough.sort_values("website_share").head(10)


In [ ]:
website_by_borough.sort_values("postcode_missing", ascending=False).head(10)


## Exercise 4: delete, fill, or keep


In [ ]:
print("rows before:", len(venues))
print("rows after dropna():", len(venues.dropna()))
print("rows with website:", len(venues.dropna(subset=["website"])))


In [ ]:
venues["website_status"] = venues["website"].fillna("No website recorded")
venues[["name", "website", "website_status"]].head()


## Exercise 5: join borough context


In [ ]:
borough_context = pd.read_csv(data_path("london_cultural_infrastructure_2023_borough_context.csv"))
borough_context.head()


In [ ]:
context_columns = [
    "borough_name",
    "population_2015",
    "median_household_income_2012_13",
    "ptal_score_2014",
    "website_share",
]
venues_with_context = venues.merge(
    borough_context[context_columns],
    on="borough_name",
    how="left",
)
venues_with_context.head()


In [ ]:
print("rows before:", len(venues))
print("rows after: ", len(venues_with_context))
print("missing ptal_score_2014:", venues_with_context["ptal_score_2014"].isna().sum())


In [ ]:
venues_with_context["high_ptal"] = venues_with_context["ptal_score_2014"] >= 5
(pd.crosstab(
    venues_with_context["high_ptal"],
    venues_with_context["has_website"],
    normalize="index",
) * 100).round(1)


## What this week was about


Missingness is not only a technical problem. In this file it affects what we can find online, what can be joined to other geographies, and how confidently we can compare places and venue types.
